# core

> Fetch competition data, push notebooks, and maintain library datasets on Kaggle

In [ ]:
#| default_exp core

In [ ]:
#|export
import json,shutil
from fastcore.utils import *
from requests.exceptions import HTTPError

In [ ]:
#|export
iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')

On Kaggle, credentials come from Kaggle secrets rather than the usual `~/.kaggle/kaggle.json`, so getting an authenticated API client depends on where you are. `import_kaggle` handles both cases:

In [ ]:
#|export
def import_kaggle():
    "Import kaggle API, using Kaggle secrets `kaggle_username` and `kaggle_key` if needed"
    if iskaggle:
        from kaggle_secrets import UserSecretsClient
        sec = UserSecretsClient()
        os.environ['KAGGLE_USERNAME'] = sec.get_secret("kaggle_username")
        if not os.environ['KAGGLE_USERNAME']: raise Exception("Please insert your Kaggle username and key into Kaggle secrets")
        os.environ['KAGGLE_KEY'] = sec.get_secret("kaggle_key")
    from kaggle import api
    api.authenticate()
    return api

The kaggle package authenticates when it is first imported, but silently ignores failure. Calling `api.authenticate()` again means a missing credential fails loudly here, and on Kaggle it picks up the secrets we just copied into the environment.

The API's list calls return response objects whose payload lives in an attribute, for example `competitions_list().competitions`:

In [ ]:
#|notest
api = import_kaggle()
L(api.competitions_list().competitions).attrgot('title')

['Passenger Screening Algorithm Challenge', 'Zillow Prize: Zillow’s Home Value Prediction (Zestimate)', 'Data Science Bowl 2017', 'Vesuvius Challenge - Ink Detection', 'ARC Prize 2026 - ARC-AGI-3', 'ARC Prize 2026 - ARC-AGI-2', 'Google DeepMind - Vibe Code with Gemini 3 Pro in AI Studio ', 'Red‑Teaming Challenge - OpenAI gpt-oss-20b', 'OpenAI to Z Challenge', 'ARC Prize 2026 - Paper Track', 'The Pokémon Company - PTCG AI Battle Challenge Strategy', 'LLM Prompt Recovery', 'Vesuvius Challenge - Surface Detection', 'Google - American Sign Language Fingerspelling Recognition', 'Second Annual Data Science Bowl', 'The Gemma 4 Good Hackathon', 'Measuring Progress Toward AGI - Cognitive Abilities', 'National Data Science Bowl', '2019 Data Science Bowl', 'Feedback Prize - Evaluating Student Writing']

## Competitions

In [ ]:
#|export
def setup_comp(competition, install=''):
    "Get a path to data for `competition`, downloading it if needed"
    if iskaggle:
        if install: os.system(f'pip install -Uqq {install}')
        return Path('../input')/competition
    else:
        path = Path(competition)
        api = import_kaggle()
        if not path.exists():
            import zipfile
            api.competition_download_cli(str(competition))
            zipfile.ZipFile(f'{competition}.zip').extractall(str(competition))
        return path

In [ ]:
#|notest
setup_comp('titanic')

Path('titanic')

If you pass a list of space separated modules to `install`, they'll be installed if running on Kaggle.

In [ ]:
#|export
def competition_submit(file_name, message, competition):
    "Submit `file_name` to `competition`, returning the submission response"
    api = import_kaggle()
    return api.competition_submit(file_name, message, competition)

Once you've created a submission file, submit it directly from your script or notebook. E.g:

```python
competition_submit('subm.csv', 'first try', 'titanic')
```

The response includes a `message` confirming the submission was created. Note that on Kaggle "code competitions" you must instead submit a notebook through `push_notebook`.

## Notebooks

In [ ]:
#|export
def nb_meta(user, id, title, file, competition=None, private=True, gpu=False, internet=True, linked_datasets=None):
    "Get the `dict` required for a kernel-metadata.json file"
    d = dict(id=f"{user}/{id}", title=title, code_file=file, language="python", kernel_type="notebook", is_private=private,
        enable_gpu=gpu, enable_internet=internet, keywords=[], dataset_sources=linked_datasets or [], kernel_sources=[])
    if competition: d["competition_sources"] = [f"competitions/{competition}"]
    return d

In [ ]:
nb_meta('jhoward', 'my-notebook', 'My notebook', 'my-notebook.ipynb', competition='paddy-disease-classification')

{'id': 'jhoward/my-notebook',
 'title': 'My notebook',
 'code_file': 'my-notebook.ipynb',
 'language': 'python',
 'kernel_type': 'notebook',
 'is_private': True,
 'enable_gpu': False,
 'enable_internet': True,
 'keywords': [],
 'dataset_sources': [],
 'kernel_sources': [],
 'competition_sources': ['competitions/paddy-disease-classification']}

In [ ]:
#|export
def push_notebook(user, id, title, file, path='.', competition=None, private=True, gpu=False, internet=True, linked_datasets=None):
    "Push notebook `file` to Kaggle Notebooks"
    meta = nb_meta(user, id, title, file=file, competition=competition, private=private, gpu=gpu, internet=internet, linked_datasets=linked_datasets)
    path = Path(path)
    path.mkdir(exist_ok=True, parents=True)
    with open(path/'kernel-metadata.json', 'w') as f: json.dump(meta, f, indent=2)
    api = import_kaggle()
    return api.kernels_push(str(path))

Note that Kaggle recommends that the `id` match the *slug* for the title -- i.e it should be the same as the title, but lowercase, no punctuation, and spaces replaced with dashes. E.g:

```python
push_notebook('jhoward', 'first-steps-road-to-the-top-part-1',
              title='First Steps: Road to the Top, Part 1',
              file='first-steps-road-to-the-top-part-1.ipynb',
              competition='paddy-disease-classification',
              private=False, gpu=True)
```

The response returned by Kaggle includes the notebook's `url`, and an `error` string if the push failed.

## Datasets

### Core

In [ ]:
#|export
def check_ds_exists(
    dataset_slug # Dataset slug (ie "uciml/iris")
):
    "Does `dataset_slug` exist on Kaggle?"
    api = import_kaggle()
    try: api.dataset_list_files(dataset_slug)
    except HTTPError: return False
    return True

Because it asks Kaggle directly, this works for any public dataset, not just your own. A dataset you cannot see reports as not existing:

In [ ]:
#|notest
assert not check_ds_exists('zillow/no-such-dataset')
check_ds_exists('uciml/iris')

True

In [ ]:
#|export
def mk_dataset(
    dataset_path, # Local path to create dataset in
    title, # Name of the dataset
    force=False, # Should it overwrite or error if exists?
    upload=True # Should it upload and create on kaggle
):
    "Creates minimal dataset metadata needed to push new dataset to kaggle"
    dataset_path = Path(dataset_path)
    dataset_path.mkdir(exist_ok=force,parents=True)
    api = import_kaggle()
    api.dataset_initialize(str(dataset_path))
    md = json.load(open(dataset_path/'dataset-metadata.json'))
    md['title'] = title
    md['id'] = md['id'].replace('INSERT_SLUG_HERE',title)
    json.dump(md, open(dataset_path/'dataset-metadata.json','w'), indent=2)
    if upload:
        (dataset_path/'empty.txt').touch()
        api.dataset_create_new(str(dataset_path), public=True, dir_mode='zip', quiet=True)

The `upload=False` form only creates the folder and its `dataset-metadata.json` locally, which is also how we can demonstrate it without creating a real dataset:

In [ ]:
#|notest
mk_dataset('./testds', 'mytestds', force=True, upload=False)
md = json.load(open('./testds/dataset-metadata.json'))
assert md['title'] == 'mytestds'
assert md['id'].endswith('/mytestds')
md

Data package template written to: testds/dataset-metadata.json


{'title': 'mytestds',
 'id': 'jhoward/mytestds',
 'licenses': [{'name': 'CC0-1.0'}]}

`get_dataset` downloads an existing dataset, along with its metadata file, ready to update and push back:

In [ ]:
#|export
def get_dataset(
    dataset_path, # Local path to download dataset to
    dataset_slug, # Dataset slug (ie "uciml/iris")
    unzip=True, # Should it unzip after downloading?
    force=False # Should it overwrite or error if dataset_path exists?
):
    "Downloads an existing dataset and metadata from kaggle"
    if not force: assert not Path(dataset_path).exists()
    api = import_kaggle()
    api.dataset_metadata(dataset_slug, str(dataset_path))
    api.dataset_download_files(dataset_slug, str(dataset_path), force=force, unzip=unzip)

To fill a library dataset with installable files, download the library's wheels with pip:

In [ ]:
#|export
def get_pip_library(
    dataset_path, # Local path to download pip library to
    pip_library, # name of library for pip to install
    pip_cmd="pip" # pip base to use (ie "pip3" or "pip")
):
    "Download the whl files for `pip_library` and store in `dataset_path`"
    run(f"{pip_cmd} download {pip_library} -d {dataset_path}")

`get_pip_libraries` does the same for everything in a `requirements.txt` file.

In [ ]:
#|export
def get_pip_libraries(
    dataset_path, # Local path to download pip libraries to
    requirements_path, # path to requirements file
    pip_cmd="pip" # pip base to use (ie "pip3" or "pip")
):
    "Download whl files for a `requirements.txt` file and store in `dataset_path`"
    run(f"{pip_cmd} download -r {requirements_path} -d {dataset_path}")

In [ ]:
dl_path = Path('./mylib')
get_pip_library(dl_path,'fastkaggle')
assert 1==len([o for o in dl_path.ls() if str(o).startswith(f"{dl_path}/fastkaggle")])

Once the local folder holds the new files, `push_dataset` uploads a new version:

In [ ]:
#|export
def push_dataset(
    dataset_path, # Local path where dataset is stored
    version_comment # Comment associated with this dataset update
):
    "Push dataset update to kaggle. Dataset path must contain dataset metadata file"
    api = import_kaggle()
    api.dataset_create_version(str(dataset_path), version_comment, dir_mode='zip', quiet=True)

`get_local_ds_ver` reads the version number from a library's wheel in a local dataset copy, so the high level functions below can tell whether the Kaggle copy is up to date.

In [ ]:
#|export
def get_local_ds_ver(
    lib_path, # Local path dataset is stored in
    lib # Name of library (ie "fastcore")
):
    "Checks a local copy of kaggle dataset for library version number"
    wheel_lib_name = lib.replace('-','_')
    local_path = Path(lib_path)/f"library-{lib}"
    lib_whl = local_path.ls().filter(lambda x: wheel_lib_name in x.name.lower())
    if 1==len(lib_whl): return re.search(rf"(?<={wheel_lib_name}-)[\d+.]+\d", lib_whl[0].name.lower())[0]
    elif 0<len(local_path.ls().filter(lambda x: 'dist' in x.name)):
        lib_whl = (local_path/'dist').ls().filter(lambda x: wheel_lib_name in x.name.lower())
        if 1==len(lib_whl): return re.search(rf"(?<={wheel_lib_name}-)[\d+.]+\d", lib_whl[0].name.lower())[0]
    return None

### High Level

In [ ]:
#|export
def create_libs_datasets(
    libs, # library or list of libraries to create datasets for (ie 'fastcore' or ['fastcore','fastkaggle'])
    lib_path, # Local path to dl/create dataset
    username, # Your username
    clear_after=False # Delete local copies after sync with kaggle?
):
    "For each library, create or update a kaggle dataset with the latest version"
    retain = ["dataset-metadata.json"]
    for lib in L(libs):
        title = f"library-{lib}"
        local_path = Path(lib_path)/title
        print(f"{lib} | Processing as {title} at {local_path}")
        if local_path.exists(): shutil.rmtree(local_path)

        print(f"{lib} | Downloading or Creating Dataset")
        if check_ds_exists(f"{username}/{title}"): get_dataset(local_path, f"{username}/{title}", force=True)
        else: mk_dataset(local_path, title, force=True)

        print(f"{lib} | Checking dataset version against pip")
        ver_local_orig = get_local_ds_ver(lib_path, lib)
        for item in local_path.ls():
            if item.name not in retain:
                if item.is_dir(): shutil.rmtree(item)
                else: item.unlink()
        get_pip_library(local_path, lib)
        ver_local_new = get_local_ds_ver(lib_path, lib)
        if ver_local_new != ver_local_orig or (ver_local_new is None and ver_local_orig is None):
            print(f"{lib} | Updating {lib} in Kaggle from {ver_local_orig} to {ver_local_new}")
            push_dataset(local_path, ifnone(ver_local_new, "Version Unknown"))
        else: print(f"{lib} | Kaggle dataset already up to date {ver_local_orig} to {ver_local_new}")
        if clear_after: shutil.rmtree(local_path)
        print(f"{lib} | Complete")

In [ ]:
#|export
def create_requirements_dataset(
    req_fpath, # Path to requirements.txt file
    lib_path, # Local path to dl/create dataset
    title, # Title you want the kaggle dataset named
    username, # Your username
    retain=["dataset-metadata.json"], # Files that should not be removed
    version_notes="New Update" # Comment associated with this dataset update
):
    "Download everything needed in a `requirements.txt` file to a dataset and upload to kaggle"
    local_path = Path(lib_path)/title
    print(f"Processing {title} at {local_path}")
    if local_path.exists(): shutil.rmtree(local_path)

    print("-----Downloading or Creating Dataset")
    if check_ds_exists(f"{username}/{title}"): get_dataset(local_path, f"{username}/{title}", force=True)
    else: mk_dataset(local_path, title, force=True)

    print("-----Checking dataset version against pip")
    orig_ds = local_path.ls().sorted()
    for item in local_path.ls():
        if item.name not in retain:
            if item.is_dir(): shutil.rmtree(item)
            else: item.unlink()
    get_pip_libraries(local_path, req_fpath)
    new_ds = local_path.ls().sorted()

    if orig_ds != new_ds:
        print(f"-----Updating {title} in Kaggle")
        push_dataset(local_path, version_notes)
    else: print("-----Kaggle dataset already up to date")
    print('Complete')

## Export -

In [ ]:
#|hide
import nbdev; nbdev.nbdev_export()
